**Author:** Pankaj Yadav          
**Date:** 05/2/2026                 
**Description:**              
This assignment focuses on learning fine tuning of a Large Language Model.
Its based on Matthew MacFarquhar's blog in Medium : Sculpting Language: GPT-2 Fine-Tuning with LoRa
linked at https://blog.devgenius.io/sculpting-language-gpt-2-fine-tuning-with-lora-1caf3bfbc3c6  

The author is trying to show that you can turn a general GPT-2 model into a task-specific model cheaply and practically using LoRA fine-tuning.

**In this blog, the concrete goal is:**                  
Take pretrained GPT-2.              
Fine-tune it on a quote-tagging dataset.             
Teach it a specific completion format: "quote ->: tags".              
Save LoRA-adapted weights and run inference with them.                   


**Why this matters in the post:**                
Full fine-tuning is expensive.           
LoRA updates only a small low-rank set of parameters, so training needs less VRAM/time.           
You can get useful task-specific behavior from a smaller model that can run locally, instead of relying on very large models for everything.       

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from peft import LoraConfig, get_peft_model
from datasets import load_dataset


In [2]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [3]:

model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:

# FREEZE WEIGHTS
for param in model.parameters():
    param.requires_grad = False

# LoRa
config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)


C:\Users\panka\AppData\Roaming\Python\Python312\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [5]:

# LOAD AND STURCTURE DATA
data = load_dataset("Abirate/english_quotes")


def merge_columns(entry):
    entry["prediction"] = entry["quote"] + " ->: " + str(entry["tags"])
    return entry


data['train'] = data['train'].map(merge_columns)
print(data['train']['prediction'][:5])

data = data.map(lambda samples: tokenizer(samples['prediction']), batched=True)

["“Be yourself; everyone else is already taken.” ->: ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']", "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.” ->: ['best', 'life', 'love', 'mistakes', 'out-of-control', 'truth', 'worst']", "“Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.” ->: ['human-nature', 'humor', 'infinity', 'philosophy', 'science', 'stupidity', 'universe']", "“So many books, so little time.” ->: ['books', 'humor']", "“A room without books is like a body without a soul.” ->: ['books', 'simile', 'soul']"]


In [6]:

def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )


print_trainable_parameters(model)


trainable params: 589824 || all params: 125029632 || trainable%: 0.4717473694555863


In [7]:

# TRAINING
trainer = transformers.Trainer(
    model=model,
    train_dataset=data['train'],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        max_steps=500,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir='outputs',
        auto_find_batch_size=True
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
model.config.use_cache = False
trainer.train()

torch.save(model.state_dict(), 'lora.pt')

C:\Users\panka\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,4.512040
2,4.308538
3,4.263571
4,4.363004
5,4.067844
6,4.375667
7,4.371228
8,4.427879
9,4.072722
10,4.406124


In [9]:

model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    device_map='auto',
)

config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [10]:

model = get_peft_model(model, config)
model = model.to(device)
model.load_state_dict(torch.load("lora.pt", map_location=device))

C:\Users\panka\AppData\Roaming\Python\Python312\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


<All keys matched successfully>

In [11]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [12]:
tokenizer.pad_token = tokenizer.eos_token

In [13]:
with torch.no_grad():
    batch = tokenizer("“Life is like a box of chocolates, you never know what you are gonna get” ->: ", return_tensors='pt').to(device)
    output_tokens = model.generate(**batch, max_new_tokens=25)

print('\n\n', tokenizer.decode(output_tokens[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.




 “Life is like a box of chocolates, you never know what you are gonna get” ->: vernacular-life, 'life', 'life-inspirational', 'life-inspirational-inspirational
